# ACFD-Transformer — Reproducible Pipeline (Google Colab)

Reference implementation for **"Optimizing APT Attack Detection with Long-Sequence Transformer Variants and Attention-based Contextual Diffusion."**

**Dataset:** the *network-traffic* portion of CICAPT-IIoT (`phase1_NetworkData.csv` + `phase2_NetworkData.csv`). A High-RAM + GPU runtime is recommended.

**Pipeline:**
1. Read the two CSV files in chunks, keep every real attack flow, sub-sample benign flows, and cache the processed arrays to a compressed `.npz` (subsequent runs load in seconds).
2. For each seed: 70/15/15 split on real data → Min-Max scaler fit on train only → sliding window (label = terminal flow) → ACFD (conditional DDPM, T=1000) augmenting the **training split only** → Longformer (2 layers, 8 heads, d=128) → early stopping on validation F1 → evaluation on a **100% real** test set.
3. Three seeds → mean ± standard deviation.
4. SHAP feature importance (GradientExplainer) and a diffusion-process histogram, both on real data.
5. Ablation (`use_acfd=False`) for the With/Without-ACFD comparison.

Synthetic samples are used in training only; the validation and test sets contain real data exclusively.


In [ ]:
# ===== Cell 1: Setup & Config =====
!pip install -q transformers shap

# If running on Google Colab with the dataset on Google Drive, mount it:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

import os, math, json, random, time, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, classification_report, confusion_matrix)
from transformers import LongformerConfig, LongformerModel

class CFG:
    # Update these paths to point to the two CICAPT-IIoT network-traffic CSV files.
    PHASE1   = '/content/drive/My Drive/CICAPT-IIoT/phase1_NetworkData.csv'
    PHASE2   = '/content/drive/My Drive/CICAPT-IIoT/phase2_NetworkData.csv'
    OUT_DIR  = '/content/drive/My Drive/CICAPT-IIoT/acfd_outputs'
    CACHE    = '/content/drive/My Drive/CICAPT-IIoT/acfd_outputs/cicapt_network_cache.npz'

    N_FEATURES    = 63          # informative numeric features (Section 4.2)
    TARGET_BENIGN = 50_000      # benign flows sampled from both phases
    CHUNKSIZE     = 1_000_000   # CSV is read in chunks to bound memory

    WINDOW   = 10               # sliding-window size W
    # ACFD diffusion module (Table 3)
    T_STEPS  = 1000
    BETA_START, BETA_END = 1e-4, 0.02
    ACFD_HIDDEN, ACFD_EPOCHS, ACFD_LR = 128, 100, 1e-3
    # Longformer backbone (Table 3)
    EMBED_DIM, N_LAYERS, N_HEADS, FFN_DIM, DROPOUT = 128, 2, 8, 512, 0.1
    # Training (Section 3.5)
    BATCH_SIZE, LR, MAX_EPOCHS, PATIENCE = 64, 1e-4, 50, 7

    SEEDS = [42, 43, 44]        # multiple seeds for mean +/- std

os.makedirs(CFG.OUT_DIR, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

In [ ]:
# ===== Cell 2: Load data (chunked) + cache =====
# Columns that are identifiers / labels rather than behavioural features.
META_COLS = {'ts', 'Source IP', 'Destination IP', 'Source Port', 'Destination Port',
             'Protocol_name', 'label', 'subLabel', 'subLabelCat', 'Flow ID', 'id'}

def attack_mask_from(sub):
    s = sub.astype(str).str.strip()
    return (s != '0') & (s != '0.0') & (s != '') & (s.str.lower() != 'nan')

if os.path.exists(CFG.CACHE):
    print('Found cache -> loading in a few seconds...')
    z = np.load(CFG.CACHE, allow_pickle=True)
    X_all, y_all, sub_codes = z['X'], z['y'], z['sub']
    FEATURES, SUB_NAMES = list(z['features']), list(z['sub_names'])
else:
    print('No cache yet -> reading the two CSV files in chunks (first run ~10-20 min)...')
    head = pd.read_csv(CFG.PHASE2, nrows=5)
    print(f'phase2 has {len(head.columns)} columns:', list(head.columns))
    feat_candidates = [c for c in head.columns if c not in META_COLS]

    rng = np.random.default_rng(123)            # fixed seed for the data-collection step
    attack_parts, benign_parts = [], []
    benign_cap = CFG.TARGET_BENIGN * 3          # over-collect, then sub-sample exactly

    def harvest(path, phase):
        """Keep every attack row; sample a small fraction of benign per chunk."""
        n_rows = 0
        usecols = feat_candidates + [c for c in ('subLabelCat',) if c in head.columns]
        dtypes = {'subLabelCat': 'str'} if 'subLabelCat' in head.columns else None
        for chunk in pd.read_csv(path, chunksize=CFG.CHUNKSIZE,
                                 usecols=lambda c: c in set(usecols), dtype=dtypes):
            n_rows += len(chunk)
            if phase == 2 and 'subLabelCat' in chunk.columns:
                m = attack_mask_from(chunk['subLabelCat'])
                if m.any():
                    attack_parts.append(chunk[m].copy())
                ben = chunk[~m]
            else:
                ben = chunk
            if sum(len(b) for b in benign_parts) < benign_cap and len(ben):
                take = max(1, int(len(ben) * 0.005))
                idx = rng.choice(len(ben), size=min(take, len(ben)), replace=False)
                benign_parts.append(ben.iloc[idx].copy())
            del chunk; gc.collect()
        print(f'   phase{phase}: scanned {n_rows:,} rows')

    t0 = time.time()
    harvest(CFG.PHASE2, 2)
    harvest(CFG.PHASE1, 1)
    print(f'Reading finished in {(time.time()-t0)/60:.1f} min')

    attack_df = pd.concat(attack_parts, ignore_index=True)
    benign_df = pd.concat(benign_parts, ignore_index=True)
    if len(benign_df) > CFG.TARGET_BENIGN:
        benign_df = benign_df.sample(n=CFG.TARGET_BENIGN, random_state=123)
    print(f'Real attack flows: {len(attack_df):,} | benign sampled: {len(benign_df):,}')

    # Attack sub-type as the ACFD conditioning signal: 0 = benign, 1..K = attack type
    sub_attack, SUB_NAMES = pd.factorize(attack_df['subLabelCat'].astype(str))
    sub_codes = np.concatenate([np.zeros(len(benign_df), dtype=np.int64), sub_attack + 1])
    SUB_NAMES = ['benign'] + list(SUB_NAMES)

    df = pd.concat([benign_df, attack_df], ignore_index=True)
    y_all = np.concatenate([np.zeros(len(benign_df), dtype=np.int64),
                            np.ones(len(attack_df), dtype=np.int64)])

    # Feature selection: coerce numeric, drop constant columns, keep top variance
    num = df[[c for c in feat_candidates if c in df.columns]].apply(pd.to_numeric, errors='coerce')
    num = num.replace([np.inf, -np.inf], np.nan).fillna(0)
    num = num.loc[:, num.nunique() > 1]
    print(f'Usable numeric features: {num.shape[1]}')
    FEATURES = num.var().sort_values(ascending=False).index[:CFG.N_FEATURES].tolist()
    X_all = num[FEATURES].values.astype(np.float32)

    np.savez_compressed(CFG.CACHE, X=X_all, y=y_all, sub=sub_codes,
                        features=np.array(FEATURES), sub_names=np.array(SUB_NAMES, dtype=object))
    print('Cached processed dataset ->', CFG.CACHE)
    del df, attack_df, benign_df, num; gc.collect()

print(f'\nReal dataset: X={X_all.shape} | benign={(y_all==0).sum():,} | attack={(y_all==1).sum():,}')
print(f'{len(FEATURES)} features:')
for i in range(0, len(FEATURES), 5): print('  ', FEATURES[i:i+5])
print('Sub-types:', SUB_NAMES)

In [ ]:
# ===== Cell 3: Model definitions (ACFD + Longformer) and helpers =====
class SinusoidalTimeEmb(nn.Module):
    def __init__(self, dim): super().__init__(); self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        a = t[:, None].float() * freqs[None]
        return torch.cat([a.sin(), a.cos()], dim=-1)

class ACFDDenoiser(nn.Module):
    """eps_theta(x_t, t, c): 3-layer MLP (128 units, SiLU), conditioned on the
    attack sub-type c (Eq. 4)."""
    def __init__(self, x_dim, n_cond, t_dim=64, c_dim=64, hidden=CFG.ACFD_HIDDEN):
        super().__init__()
        self.t_emb = SinusoidalTimeEmb(t_dim)
        self.c_emb = nn.Embedding(n_cond, c_dim)
        self.net = nn.Sequential(
            nn.Linear(x_dim + t_dim + c_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, x_dim))
    def forward(self, x, t, c):
        return self.net(torch.cat([x, self.t_emb(t), self.c_emb(c)], dim=-1))

class ACFD:
    """Conditional DDPM: T=1000, linear beta schedule 1e-4 -> 0.02 (Table 3)."""
    def __init__(self, x_dim, n_cond):
        self.T = CFG.T_STEPS
        self.betas  = torch.linspace(CFG.BETA_START, CFG.BETA_END, self.T, device=device)
        self.alphas = 1.0 - self.betas
        self.abar   = torch.cumprod(self.alphas, dim=0)
        self.model  = ACFDDenoiser(x_dim, n_cond).to(device)
    def pretrain(self, X, c, epochs=CFG.ACFD_EPOCHS, bs=256, lr=CFG.ACFD_LR, verbose=True):
        opt = torch.optim.AdamW(self.model.parameters(), lr=lr)
        dl = DataLoader(TensorDataset(X, c), batch_size=bs, shuffle=True)
        self.model.train()
        for ep in range(epochs):
            tot = 0.0
            for xb, cb in dl:
                xb, cb = xb.to(device), cb.to(device)
                t = torch.randint(0, self.T, (len(xb),), device=device)
                eps = torch.randn_like(xb)
                ab = self.abar[t][:, None]
                xt = ab.sqrt() * xb + (1 - ab).sqrt() * eps
                loss = nn.functional.mse_loss(self.model(xt, t, cb), eps)
                opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()
            if verbose and (ep + 1) % 20 == 0:
                print(f'      [ACFD] epoch {ep+1}/{epochs} noise-MSE={tot/len(dl):.4f}')
    @torch.no_grad()
    def sample(self, n, cond_pool, x_dim, bs=2048):
        self.model.eval(); out = []
        for s in range(0, n, bs):
            m = min(bs, n - s)
            x = torch.randn(m, x_dim, device=device)
            c = cond_pool[torch.randint(0, len(cond_pool), (m,))].to(device)
            for t in reversed(range(self.T)):
                tt = torch.full((m,), t, dtype=torch.long, device=device)
                eps = self.model(x, tt, c)
                a, ab, b = self.alphas[t], self.abar[t], self.betas[t]
                x = (x - (1 - a) / (1 - ab).sqrt() * eps) / a.sqrt()
                if t > 0: x = x + b.sqrt() * torch.randn_like(x)
            out.append(x.clamp(0, 1).cpu())   # features are Min-Max scaled to [0,1]
        return torch.cat(out)

class LongformerAPTDetector(nn.Module):
    """HuggingFace Longformer: 2 layers, 8 heads, d=128 (Table 3). vocab_size is set
    to a minimal value because inputs_embeds are fed directly, avoiding the large
    unused word-embedding table."""
    def __init__(self, input_dim, embed_dim=CFG.EMBED_DIM):
        super().__init__()
        self.embedding = nn.Linear(input_dim, embed_dim)
        cfg = LongformerConfig(
            vocab_size=4, hidden_size=embed_dim, num_hidden_layers=CFG.N_LAYERS,
            num_attention_heads=CFG.N_HEADS, intermediate_size=CFG.FFN_DIM,
            attention_window=[CFG.WINDOW] * CFG.N_LAYERS,   # spans the full window W=10
            max_position_embeddings=CFG.WINDOW + 8,
            hidden_dropout_prob=CFG.DROPOUT, attention_probs_dropout_prob=CFG.DROPOUT,
            pad_token_id=0, bos_token_id=1, eos_token_id=2, sep_token_id=2)
        self.transformer = LongformerModel(cfg, add_pooling_layer=False)
        self.head = nn.Sequential(nn.Linear(embed_dim, 64), nn.ReLU(),
                                  nn.Dropout(CFG.DROPOUT), nn.Linear(64, 2))
    def forward(self, x):
        h = self.transformer(inputs_embeds=self.embedding(x)).last_hidden_state
        return self.head(h.mean(dim=1))   # global average pooling + MLP head

def make_windows(X, y, sub=None, w=CFG.WINDOW):
    """Window = rows i..i+w-1; label = the LAST flow of the window."""
    xs = np.lib.stride_tricks.sliding_window_view(X, (w, X.shape[1]))[:, 0]
    ys = y[w - 1:]
    out = [torch.tensor(xs.copy(), dtype=torch.float32), torch.tensor(ys, dtype=torch.long)]
    if sub is not None: out.append(torch.tensor(sub[w - 1:], dtype=torch.long))
    return out

@torch.no_grad()
def evaluate(model, dl):
    model.eval(); ys, ps, probs = [], [], []
    for xb, yb in dl:
        logits = model(xb.to(device))
        ys.append(yb); ps.append(logits.argmax(-1).cpu())
        probs.append(logits.softmax(-1)[:, 1].cpu())
    return torch.cat(ys).numpy(), torch.cat(ps).numpy(), torch.cat(probs).numpy()

_tmp = LongformerAPTDetector(len(FEATURES))
N_PARAMS = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
del _tmp
print(f'Longformer detector parameters: {N_PARAMS:,} ({N_PARAMS/1e6:.2f}M)')

In [ ]:
# ===== Cell 4: run_experiment(seed, use_acfd) =====
def run_experiment(seed, use_acfd=True, keep_artifacts=False):
    set_seed(seed)
    print(f'\n{"="*64}\n  SEED {seed} | ACFD = {use_acfd}\n{"="*64}')

    # 70/15/15 split on REAL data
    X_tr, X_tmp, y_tr, y_tmp, s_tr, _ = train_test_split(
        X_all, y_all, sub_codes, test_size=0.30, stratify=y_all, random_state=seed)
    X_va, X_te, y_va, y_te = train_test_split(
        X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=seed)[:4]

    scaler = MinMaxScaler().fit(X_tr)                      # fit on TRAIN only
    X_tr, X_va, X_te = scaler.transform(X_tr), scaler.transform(X_va), scaler.transform(X_te)

    Xw_tr, yw_tr, sw_tr = make_windows(X_tr, y_tr, s_tr)
    Xw_va, yw_va = make_windows(X_va, y_va)[:2]
    Xw_te, yw_te = make_windows(X_te, y_te)[:2]
    print(f'Windows: train {tuple(Xw_tr.shape)} | val {tuple(Xw_va.shape)} | test {tuple(Xw_te.shape)}'
          f' | train APT-rate {yw_tr.float().mean():.4f}')

    # Sanity check: the data must carry a learnable signal
    clf = LogisticRegression(max_iter=1000, class_weight='balanced')
    clf.fit(Xw_tr[:, -1, :].numpy(), yw_tr.numpy())
    print(f'[Sanity] LogReg val F1 = {f1_score(yw_va.numpy(), clf.predict(Xw_va[:, -1, :].numpy())):.4f}')

    # ACFD augmentation on the TRAIN split ONLY
    if use_acfd:
        x_dim = CFG.WINDOW * len(FEATURES)
        mask = yw_tr == 1
        cond_pool = sw_tr[mask]
        n_cond = len(SUB_NAMES)
        print(f'Pre-training ACFD on {int(mask.sum())} real attack windows, {n_cond} conditions...')
        acfd = ACFD(x_dim, n_cond)
        acfd.pretrain(Xw_tr[mask].reshape(-1, x_dim), cond_pool)
        n_need = int((yw_tr == 0).sum() - (yw_tr == 1).sum())
        if n_need > 0:
            print(f'Sampling {n_need:,} synthetic minority windows (T={CFG.T_STEPS})...')
            t0 = time.time()
            synth = acfd.sample(n_need, cond_pool, x_dim).reshape(-1, CFG.WINDOW, len(FEATURES))
            print(f'   done in {(time.time()-t0)/60:.1f} min')
            Xw_bal = torch.cat([Xw_tr, synth])
            yw_bal = torch.cat([yw_tr, torch.ones(len(synth), dtype=torch.long)])
        else:
            Xw_bal, yw_bal = Xw_tr, yw_tr
    else:
        acfd, Xw_bal, yw_bal = None, Xw_tr, yw_tr

    # Train the Longformer detector
    model = LongformerAPTDetector(len(FEATURES)).to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=CFG.LR)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG.MAX_EPOCHS)
    crit  = nn.CrossEntropyLoss()
    train_dl = DataLoader(TensorDataset(Xw_bal, yw_bal), batch_size=CFG.BATCH_SIZE, shuffle=True)
    val_dl   = DataLoader(TensorDataset(Xw_va, yw_va), batch_size=1024)

    best_f1, best_state, patience = -1.0, None, CFG.PATIENCE
    for epoch in range(1, CFG.MAX_EPOCHS + 1):
        model.train(); tot = 0.0
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); loss = crit(model(xb), yb); loss.backward(); opt.step()
            tot += loss.item()
        sched.step()
        yv, pv, _ = evaluate(model, val_dl)
        vf1 = f1_score(yv, pv)
        print(f'  Epoch {epoch:02d} | loss {tot/len(train_dl):.4f} | val F1 {vf1:.4f} | pred-APT {pv.mean():.3f}')
        if vf1 > best_f1:
            best_f1, patience = vf1, CFG.PATIENCE
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience -= 1
            if patience == 0: print(f'  Early stop (best val F1 {best_f1:.4f})'); break
    model.load_state_dict(best_state)

    # Evaluate on the REAL test set
    test_dl = DataLoader(TensorDataset(Xw_te, yw_te), batch_size=1024)
    y, p, prob = evaluate(model, test_dl)
    m = {'seed': seed, 'use_acfd': use_acfd,
         'accuracy_%': accuracy_score(y, p) * 100,
         'precision': precision_score(y, p, zero_division=0),
         'recall': recall_score(y, p, zero_division=0),
         'f1': f1_score(y, p, zero_division=0),
         'auc_roc': roc_auc_score(y, prob)}
    print('  TEST: ' + ' | '.join(f'{k}={v:.4f}' for k, v in m.items() if isinstance(v, float)))
    print(confusion_matrix(y, p))

    arts = None
    if keep_artifacts:
        arts = {'model': model, 'acfd': acfd, 'scaler': scaler,
                'Xw_tr': Xw_tr, 'yw_tr': yw_tr, 'Xw_te': Xw_te, 'yw_te': yw_te}
    else:
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    return m, arts

In [ ]:
# ===== Cell 5: Main run — multi-seed, with ACFD (Table 4, Longformer row) =====
results, ART = [], None
for i, seed in enumerate(CFG.SEEDS):
    m, a = run_experiment(seed, use_acfd=True, keep_artifacts=(i == 0))
    results.append(m)
    if a is not None: ART = a

keys = ['accuracy_%', 'precision', 'recall', 'f1', 'auc_roc']
print('\n' + '=' * 64)
print(f'RESULTS over {len(CFG.SEEDS)} seeds - ACFD-Transformer (Longformer), {N_PARAMS/1e6:.2f}M params')
print('=' * 64)
summary = {}
for k in keys:
    vals = np.array([r[k] for r in results])
    summary[k] = {'mean': float(vals.mean()), 'std': float(vals.std(ddof=1) if len(vals) > 1 else 0)}
    print(f'  {k:12s}: {vals.mean():.4f} +/- {vals.std(ddof=1) if len(vals)>1 else 0:.4f}   (seeds: {np.round(vals,4)})')

with open(os.path.join(CFG.OUT_DIR, 'table4_longformer_real.json'), 'w') as f:
    json.dump({'per_seed': results, 'summary': summary, 'params_M': N_PARAMS/1e6,
               'n_features': len(FEATURES), 'features': FEATURES,
               'real_benign': int((y_all==0).sum()), 'real_attack': int((y_all==1).sum())}, f, indent=2)
torch.save(ART['model'].state_dict(), os.path.join(CFG.OUT_DIR, 'acfd_longformer_seed42.pth'))
print('\nSaved metrics and checkpoint to', CFG.OUT_DIR)

In [ ]:
# ===== Cell 6: SHAP feature importance (GradientExplainer, real test data) =====
import shap

model = ART['model'].eval()
bg_idx = torch.randperm(len(ART['Xw_tr']))[:100]
te_idx = torch.randperm(len(ART['Xw_te']))[:300]
background = ART['Xw_tr'][bg_idx].to(device)
explain_x  = ART['Xw_te'][te_idx].to(device)

print('Running GradientExplainer (a few minutes)...')
explainer = shap.GradientExplainer(model, background)
sv = explainer.shap_values(explain_x)
sv = sv[1] if isinstance(sv, list) else np.array(sv)[..., 1]   # APT class
feat_imp = np.abs(np.array(sv)).mean(axis=(0, 1))               # mean |SHAP| over samples & timesteps

order = np.argsort(feat_imp)[::-1][:15]
plt.figure(figsize=(10, 7))
plt.barh(range(len(order)), feat_imp[order][::-1], color='steelblue')
plt.yticks(range(len(order)), [FEATURES[i] for i in order][::-1])
plt.xlabel('Mean |SHAP value| (APT class)')
plt.title('Feature Importance - SHAP (GradientExplainer, real test data)')
plt.tight_layout()
for ext in ('png', 'pdf'):
    plt.savefig(os.path.join(CFG.OUT_DIR, f'fig_shap.{ext}'), dpi=300, bbox_inches='tight')
plt.show()

with open(os.path.join(CFG.OUT_DIR, 'shap_values.json'), 'w') as f:
    json.dump({FEATURES[i]: float(feat_imp[i]) for i in order}, f, indent=2)
print('Saved SHAP figure and values to', CFG.OUT_DIR)

In [ ]:
# ===== Cell 7: Diffusion-process histogram on real data =====
acfd = ART['acfd']
mask = ART['yw_tr'] == 1
x_dim = CFG.WINDOW * len(FEATURES)
real_attack = ART['Xw_tr'][mask].reshape(-1, x_dim)

# Illustrate with the highest-|SHAP| feature (at the terminal timestep)
fidx = int(np.argsort(feat_imp)[::-1][0])
col = (CFG.WINDOW - 1) * len(FEATURES) + fidx

orig = real_attack[:, col].numpy()
t_mid = CFG.T_STEPS // 2
ab = acfd.abar[t_mid].item()
noised = math.sqrt(ab) * orig + math.sqrt(1 - ab) * np.random.randn(len(orig))
n_syn = min(2000, len(orig) * 4)
synth = acfd.sample(n_syn, torch.zeros(1, dtype=torch.long) + 1, x_dim)[:, col].numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, data, title, color in zip(
        axes, [orig, noised, synth],
        [f'(a) Original Attack Data - {FEATURES[fidx]}', f'(b) Noised (t={t_mid}/{CFG.T_STEPS})', '(c) ACFD Synthetic'],
        ['#2E86AB', '#A23B72', '#F18F01']):
    ax.hist(data, bins=40, alpha=0.8, color=color, edgecolor='black')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Feature value'); ax.set_ylabel('Frequency')
plt.suptitle('ACFD Diffusion Process (real data)', fontweight='bold')
plt.tight_layout()
for ext in ('png', 'pdf'):
    plt.savefig(os.path.join(CFG.OUT_DIR, f'fig_diffusion.{ext}'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved diffusion figure to', CFG.OUT_DIR)

In [ ]:
# ===== Cell 8: Ablation — With vs. Without ACFD (Table 5) =====
abl, _ = run_experiment(CFG.SEEDS[0], use_acfd=False)

print('\n' + '=' * 64)
print('TABLE 5 - Impact of the ACFD module (same seed)')
print('=' * 64)
with_acfd = results[0]   # same seed
print(f"{'Configuration':32s} {'Acc.(%)':>8s} {'Prec.':>7s} {'Rec.':>7s} {'F1':>7s}")
print(f"{'Without ACFD (Imbalanced)':32s} {abl['accuracy_%']:8.2f} {abl['precision']:7.4f} {abl['recall']:7.4f} {abl['f1']:7.4f}")
print(f"{'With ACFD (Balanced)':32s} {with_acfd['accuracy_%']:8.2f} {with_acfd['precision']:7.4f} {with_acfd['recall']:7.4f} {with_acfd['f1']:7.4f}")

with open(os.path.join(CFG.OUT_DIR, 'table5_ablation_real.json'), 'w') as f:
    json.dump({'without_acfd': abl, 'with_acfd': with_acfd}, f, indent=2)
print('\nSaved ablation to', CFG.OUT_DIR)